# Construção da Base Analítica

Este notebook consolida a população elegível da Avaliação da Alfabetização com as fontes externas selecionadas e validadas na etapa anterior.

São incorporadas três dimensões de contexto:

- socioeconômica: INSE;
- educacional: Censo Escolar;
- econômica e demográfica: PIB e população municipal.

As transformações e regras de integração utilizadas neste notebook foram definidas previamente no notebook de seleção e validação das fontes.

O resultado final será persistido na camada Gold na tabela:

`workspace.gold.ml_alfabetizacao_aluno`

Essa tabela será utilizada como fonte para as etapas de preparação dos dados, treinamento, validação e teste dos modelos de Machine Learning.

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebbok e feita a configuração para consumir as tabelas do Base de Dados

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

## Fontes

In [0]:
dbutils.widgets.text(
    "billing_project_id",
    "fiap-tech-challenge2",
    "Billing Project GCP"
)

billing_project_id = dbutils.widgets.get("billing_project_id")

credencial_gcp = dbutils.secrets.get(
    scope="fiap-tech-challenge",
    key="gcp-service-account-b64"
)

source_project = "basedosdados"

## 2. População de Modelagem

A construção da base analítica parte da tabela Silver de alunos produzida na fase anterior do projeto.

São mantidos somente alunos:

- presentes na avaliação;
- com prova preenchida;
- pertencentes às redes Estadual ou Municipal.

Essa definição evita classificar alunos não avaliados como não alfabetizados.

A unidade analítica da base final corresponde a um aluno em determinado ano de avaliação, representado pela combinação `ano + id_aluno`.

In [0]:
df_alunos = spark.table("workspace.silver.alunos")

df_populacao_modelagem = df_alunos.filter(
    (F.col("presenca") == "1")
    & (F.col("preenchimento_caderno") == "1")
    & (F.col("rede").isin("2", "3"))
).select(
    "ano",
    "id_aluno",
    "id_escola",
    "id_municipio",
    "rede",
    "rede_descricao",
    "peso_aluno",
    F.col("alfabetizado").cast("int").alias("alfabetizado"),
)

In [0]:
display(
    df_populacao_modelagem.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_aluno").alias("qtd_ids_alunos"),
    )
    .orderBy("ano")
)

## 3. Contexto Socioeconômico - INSE

O Indicador de Nível Socioeconômico é utilizado para representar o contexto socioeconômico educacional.

Como o identificador de escola da base de alfabetização é mascarado, o indicador é agregado por município e rede.

É utilizada a edição de 2021 do INSE para ambas as avaliações, por se tratar de uma edição disponível anteriormente aos períodos analisados.

A média do indicador é ponderada pela quantidade de alunos utilizada no cálculo do INSE de cada escola.

In [0]:
tabela_inse = f"{source_project}.br_inep_indicador_nivel_socioeconomico.escola"

df_inse = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_inse)
)

In [0]:
df_inse_features = (
    df_inse.filter((F.col("ano") == 2021) & (F.col("rede").isin("2", "3")))
    .groupBy("id_municipio", "rede")
    .agg(
        (
            F.sum(F.col("inse") * F.col("quantidade_alunos_inse"))
            / F.sum("quantidade_alunos_inse")
        ).alias("inse_2021")
    )
)

## 4. Contexto Educacional - Censo Escolar

O Censo Escolar é utilizado para representar características de infraestrutura, recursos educacionais e estrutura de atendimento das escolas.

São consideradas apenas escolas das redes Estadual e Municipal com matrículas nos anos iniciais do Ensino Fundamental.

As características são agregadas por município e rede e ponderadas pela quantidade de matrículas nos anos iniciais.

Para preservar a temporalidade:

- Censo Escolar 2022 → avaliação de 2023;
- Censo Escolar 2023 → avaliação de 2024.

In [0]:
tabela_censo = f"{source_project}.br_inep_censo_escolar.escola"

df_censo = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_censo)
)

In [0]:
df_censo_modelagem = df_censo.filter(
    (F.col("ano").isin(2022, 2023))
    & (F.col("rede").isin("2", "3"))
    & (F.col("quantidade_matricula_fundamental_anos_iniciais") > 0)
)


peso_censo = F.col("quantidade_matricula_fundamental_anos_iniciais")

df_censo_features = (
    df_censo_modelagem.groupBy("ano", "id_municipio", "rede")
    .agg(
        (F.sum(F.col("agua_potavel") * peso_censo) / F.sum(peso_censo)).alias(
            "censo_pct_agua_potavel"
        ),
        (F.sum(F.col("energia_rede_publica") * peso_censo) / F.sum(peso_censo)).alias(
            "censo_pct_energia_rede_publica"
        ),
        (F.sum(F.col("esgoto_rede_publica") * peso_censo) / F.sum(peso_censo)).alias(
            "censo_pct_esgoto_rede_publica"
        ),
        (
            F.sum(F.col("biblioteca_sala_leitura") * peso_censo) / F.sum(peso_censo)
        ).alias("censo_pct_biblioteca_sala_leitura"),
        (
            F.sum(F.col("laboratorio_informatica") * peso_censo) / F.sum(peso_censo)
        ).alias("censo_pct_laboratorio_informatica"),
        (F.sum(F.col("internet_aprendizagem") * peso_censo) / F.sum(peso_censo)).alias(
            "censo_pct_internet_aprendizagem"
        ),
        (
            F.sum("quantidade_docente_fundamental_anos_iniciais")
            / F.sum("quantidade_matricula_fundamental_anos_iniciais")
            * 100
        ).alias("censo_docentes_por_100_alunos"),
        (
            F.sum("quantidade_turma_fundamental_anos_iniciais")
            / F.sum("quantidade_matricula_fundamental_anos_iniciais")
            * 100
        ).alias("censo_turmas_por_100_alunos"),
    )
    # Alinhamento com o ano da avaliação
    .withColumn("ano", F.col("ano") + 1)
)

## 5. Contexto Econômico e Demográfico

O contexto territorial é complementado por informações de PIB e população municipal.

A população representa o porte do município e também permite construir o PIB per capita.

Devido à elevada assimetria dessas variáveis, são produzidas versões logarítmicas para utilização posterior na modelagem.

Para preservar a temporalidade:

- indicadores municipais de 2022 → avaliação de 2023;
- indicadores municipais de 2023 → avaliação de 2024.

In [0]:
tabela_pib = f"{source_project}.br_ibge_pib.municipio"

df_pib = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_pib)
)

In [0]:
tabela_populacao = f"{source_project}.br_ibge_populacao.municipio"

df_populacao = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_populacao)
)

df_contexto_features = (
    df_pib.filter(F.col("ano").isin(2022, 2023))
    .select("ano", "id_municipio", "pib")
    .join(
        df_populacao.filter(F.col("ano").isin(2022, 2023)).select(
            "ano", "id_municipio", "populacao"
        ),
        on=["ano", "id_municipio"],
        how="inner",
    )
    .withColumn("pib_per_capita", F.col("pib").cast("double") / F.col("populacao"))
    .withColumn("log_pib_per_capita", F.log1p("pib_per_capita"))
    .withColumn("log_populacao", F.log1p("populacao"))
    # Alinhamento com o ano da avaliação
    .withColumn("ano", F.col("ano") + 1)
)

## 6. Integração das Fontes

As três fontes externas são integradas à população individual utilizando as chaves compatíveis com cada granularidade.

- INSE: `id_municipio + rede`;
- Censo Escolar: `ano + id_municipio + rede`;
- PIB e população: `ano + id_municipio`.

Todos os joins são realizados à esquerda para preservar integralmente a população de modelagem.

In [0]:
df_base_analitica = (
    df_populacao_modelagem.join(
        df_inse_features, on=["id_municipio", "rede"], how="left"
    )
    .join(df_censo_features, on=["ano", "id_municipio", "rede"], how="left")
    .join(df_contexto_features, on=["ano", "id_municipio"], how="left")
)

df_base_analitica = df_base_analitica.select(
    # Identificação e target
    "ano",
    "id_aluno",
    "id_escola",
    "id_municipio",
    "rede",
    "rede_descricao",
    "peso_aluno",
    "alfabetizado",

    # Contexto socioeconômico
    "inse_2021",

    # Contexto educacional
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    
    # Contexto econômico e demográfico
    "pib",
    "populacao",
    "pib_per_capita",
    "log_pib_per_capita",
    "log_populacao",
)

## 7. Validação da Base Analítica

Antes da persistência na camada Gold, são realizadas validações de volume, unicidade e completude.

A integração das fontes externas não deve alterar a quantidade original de alunos da população de modelagem.

Também é verificada a presença de valores ausentes nas features incorporadas.

In [0]:
display(
    df_base_analitica.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_aluno").alias("qtd_alunos"),
    )
    .orderBy("ano")
)

In [0]:
display(
    df_base_analitica
    .agg(
        F.count("*").alias("qtd_registros"),

        F.countDistinct(
            F.struct(
                "ano",
                "id_aluno"
            )
        ).alias("qtd_chaves")
    )
)

In [0]:
features_gold = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "pib",
    "populacao",
    "pib_per_capita",
    "log_pib_per_capita",
    "log_populacao",
]

total_gold = df_base_analitica.count()

resultado_nulos = df_base_analitica.agg(
    *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in features_gold]
).collect()[0]

nulos_gold = [
    (
        coluna,
        resultado_nulos[coluna],
        round(resultado_nulos[coluna] / total_gold * 100, 3),
    )
    for coluna in features_gold
]

df_nulos_gold = spark.createDataFrame(
    nulos_gold, ["variavel", "qtd_nulos", "pct_nulos"]
)

display(df_nulos_gold.orderBy(F.desc("pct_nulos")))

## 8. Persistência da Base Analítica

Após as validações de volume, granularidade e completude, a base consolidada é persistida em formato Delta na camada Gold.

A tabela representa a fonte única para as etapas posteriores de Machine Learning.

A construção é realizada de forma integral e determinística a partir das fontes Silver e externas. O formato Delta mantém o histórico transacional das execuções.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

(
    df_base_analitica.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable("workspace.gold.ml_alfabetizacao_aluno")
)

In [0]:
df_gold = spark.table(
    "workspace.gold.ml_alfabetizacao_aluno"
)

display(
    df_gold
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_aluno").alias("qtd_alunos")
    )
    .orderBy("ano")
)